# Create HRZZ Awards (Hrvatska Zaklada za Znanost, Croatia)

HRZZ awards from the **OpenAIRE API** (same proven adapter as FWF/NWO; `scripts/local/hrzz_to_s3.py` just sets `funder=HRZZ`).

**Prerequisites:** run `scripts/local/hrzz_to_s3.py` to download + upload first.
**Data source:** OpenAIRE **Graph API v1** (https://api.openaire.eu/graph/v1/projects?fundingShortName=HRZZ) — ~2,392 HRZZ projects via cursor pagination. **NB:** the legacy `/search/projects` API caps at 10,000 results, which silently truncates large funders (the existing `openaire_fwf` ingest is capped at ~9k of FWF's ~20k for this reason) — the Graph API cursor avoids that, so this adapter gets full coverage and is the one the automation stage should use.
**S3:** `s3a://openalex-ingest/awards/hrzz/hrzz_projects.parquet`

**HRZZ funder in OpenAlex:** funder_id 4320322674 · display_name "Hrvatska Zaklada za Znanost" · ror https://ror.org/03n51vw80 · doi 10.13039/501100004488 · HR.

**Input columns from hrzz_to_s3.py:** project_code -> funder_award_id (real HRZZ grant ref, e.g. IP-2022-10-4583 research, DOK-2020-01-xxxx doctoral, MOBODL-2023-08-xxxx mobility, cited in papers); title -> display_name; funded_amount (EUR); currency; start_date/end_date; funding_program -> funder_scheme; website_url -> landing_page_url; doi; keywords (raw only).

**Notes:** OpenAIRE's HRZZ project records carry NO organization/PI relations, so lead_investigator/institution are NULL (same as the openaire_fwf pattern; the value is the real cited grant codes). description NULL (no abstract from source; keywords kept in raw). amount in EUR, 0/missing NULLed (§6.7). funding_type: HRZZ doctoral/post-doc scholarships (DOK- doctoral, MOBODL mobility, .BD) -> fellowship, else grant. provenance 'openaire_hrzz'. priority 175.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hrzz_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hrzz/hrzz_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.hrzz_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.hrzz_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hrzz_raw LIMIT 5;

## Step 2: Create HRZZ Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hrzz_awards
USING delta
AS
WITH
hrzz_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322674  -- Fundacao para a Ciencia e a Tecnologia
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.project_code)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.project_code as funder_award_id,
        CASE WHEN TRY_CAST(g.funded_amount AS DOUBLE) > 0 THEN TRY_CAST(g.funded_amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.funded_amount AS DOUBLE) > 0 THEN COALESCE(g.currency, 'EUR') ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE
            WHEN (g.project_code RLIKE '(?i)^(DOK[-]|MOBODL|MOBDOK|MOBDOL|NPOO-DOK)' OR g.funding_program RLIKE '(?i)(doctoral|young researchers|career development|fellowship|scholarship)') THEN 'fellowship'
            ELSE 'grant'
        END as funding_type,
        g.funding_program as funder_scheme,
        'openaire_hrzz' as provenance,
        -- sanity floor: OpenAIRE has some garbage dates (e.g. year "0007") -> NULL out-of-range
        CASE WHEN YEAR(TRY_TO_DATE(g.start_date,'yyyy-MM-dd')) BETWEEN 1900 AND YEAR(current_date())+1 THEN TRY_TO_DATE(g.start_date,'yyyy-MM-dd') END as start_date,
        CASE WHEN YEAR(TRY_TO_DATE(g.end_date,'yyyy-MM-dd')) BETWEEN 1900 AND YEAR(current_date())+5 THEN TRY_TO_DATE(g.end_date,'yyyy-MM-dd') END as end_date,
        CASE WHEN YEAR(TRY_TO_DATE(g.start_date,'yyyy-MM-dd')) BETWEEN 1900 AND YEAR(current_date())+1 THEN YEAR(TRY_TO_DATE(g.start_date,'yyyy-MM-dd')) END as start_year,
        CASE WHEN YEAR(TRY_TO_DATE(g.end_date,'yyyy-MM-dd')) BETWEEN 1900 AND YEAR(current_date())+5 THEN YEAR(TRY_TO_DATE(g.end_date,'yyyy-MM-dd')) END as end_year,
        -- OpenAIRE HRZZ project records expose no PI/org -> lead_investigator NULL
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >) as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >>) as investigators,
        COALESCE(g.website_url, CONCAT('https://api.openaire.eu/search/projects?format=json&funder=HRZZ&keywords=', g.project_code)) as landing_page_url,
        g.doi as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.project_code)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.hrzz_raw g
    CROSS JOIN hrzz_funder f
    WHERE g.project_code IS NOT NULL AND TRIM(CAST(g.project_code AS STRING)) != ''
)
SELECT * FROM awards_transformed;


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'openaire_hrzz' AND priority = 175;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    175 as priority
FROM openalex.awards.hrzz_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_hrzz_awards FROM openalex.awards.hrzz_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_year, end_year
FROM openalex.awards.hrzz_awards LIMIT 10;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt FROM openalex.awards.hrzz_awards GROUP BY funder.display_name ORDER BY cnt DESC;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.hrzz_awards GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt FROM openalex.awards.hrzz_awards WHERE funder_scheme IS NOT NULL GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(funder_award_id) as has_grant_ref,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_with_amount,
    ROUND(try_divide(COUNT(start_date) * 100.0, COUNT(*)), 1) as pct_with_start
FROM openalex.awards.hrzz_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.hrzz_awards WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC LIMIT 20;

In [ ]:
%sql
SELECT funder_award_id, display_name, amount FROM openalex.awards.hrzz_awards WHERE amount IS NOT NULL ORDER BY amount DESC LIMIT 10;